In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path('.').resolve().parent))

In [2]:
from src.utils.initialization import load_config

In [3]:
from pathlib import Path

cfg = load_config(Path("../configs/mini3-train.yaml"), '')
cfg.data.dataset_dir="../data/The Hague/mini"

In [4]:
from src.models.diffusion import CityJSONDiffusionModule

ckpt = "../outputs/initial-runs/mini-3/checkpoints/last.ckpt"
model = CityJSONDiffusionModule.load_from_checkpoint(ckpt)
model.to("cuda:0")

CityJSONDiffusionModule(
  (noise): GraphNoiseModel()
  (network): rEGNNTransformer(
    (mlp_in_y): Sequential(
      (0): Linear(in_features=1, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=32, bias=True)
      (3): ReLU()
    )
    (mlp_in_X): Sequential(
      (0): Linear(in_features=5, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): ReLU()
    )
    (mlp_in_E): Sequential(
      (0): Linear(in_features=3, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=32, bias=True)
      (3): ReLU()
    )
    (mlp_in_pos): PositionsMLP(
      (mlp): Sequential(
        (0): Linear(in_features=1, out_features=16, bias=True)
        (1): ReLU()
        (2): Linear(in_features=16, out_features=1, bias=True)
      )
    )
    (tf_layers): ModuleList(
      (0-3): 4 x XEyTransformerLayer(
        (self_attn): NodeEdgeBlock(
          (in_E)

In [5]:
def _load_datamodule(cfg):
    """Datamodule for the reference-distribution metrics, or None if unavailable.

    Wasserstein/MMD/novelty need the train+test splits; validity, rejection rate and
    face coherence do not. A missing dataset degrades the run instead of killing it.
    """
    from src.utils.setup_utils import create_datamodule
    
    datamodule = create_datamodule(cfg)
    datamodule.setup()
    return datamodule

In [6]:
datamodule = _load_datamodule(cfg)

In [7]:
from src.eval.sampling import draw_samples

records, stats = draw_samples(model, 1, 10)

Graph has 2 vertices and 4 faces; cannot build a geometry.
Face node 17 has only 2 member vertices, skipping.
Face node 20 has only 2 member vertices, skipping.
Face node 37 has only 1 member vertices, skipping.
Face node 38 has only 2 member vertices, skipping.
Face node 39 has only 2 member vertices, skipping.
Face node 48 has only 2 member vertices, skipping.
Face node 49 has only 0 member vertices, skipping.
Face node 5 has only 2 member vertices, skipping.
Face node 19 has only 2 member vertices, skipping.
Face node 20 has only 1 member vertices, skipping.
Face node 31 has only 2 member vertices, skipping.
Face node 48 has only 2 member vertices, skipping.
Face node 10 has only 1 member vertices, skipping.
Face node 23 has only 2 member vertices, skipping.
Face node 25 has only 1 member vertices, skipping.
Face node 45 has only 1 member vertices, skipping.
Face node 12 has only 1 member vertices, skipping.
Face node 21 has only 2 member vertices, skipping.
Face node 28 has only 2 

In [8]:
records[1]

{'coords': array([[-2.53827238,  0.01999824,  2.341712  ],
        [ 0.78677434, -1.37794042, -0.43584204],
        [ 0.11937132,  1.2970643 , -0.10174049],
        [ 1.87635338,  3.83589697, -1.35677409],
        [ 0.79894966, -0.97383153,  0.28603011],
        [ 0.84251392, -1.28948939,  0.5695489 ],
        [ 0.51161987, -1.20012331, -0.33228728],
        [-0.22867429, -0.31713596,  1.49106598],
        [ 0.2762101 , -1.0242703 ,  0.43108511],
        [ 1.4610734 ,  0.20829314,  0.16385508],
        [-0.73711771, -1.05974889,  0.32448879],
        [-2.83141565, -1.60784316, -3.87428451],
        [-0.69932759,  1.12017322,  0.22398998],
        [ 1.79605889,  3.69916892,  0.20569642],
        [ 0.75665349,  0.57094848,  0.87877399],
        [ 0.54635495,  0.97826529, -0.10116849],
        [-0.55520523, -0.78242242, -0.7330991 ],
        [ 0.43475431, -0.18878555,  1.17321765],
        [ 1.13311088, -0.733513  , -0.07947174],
        [-1.59057271,  1.99556398,  0.652098  ],
        [-

In [9]:
import numpy as np
import plotly.io as pio
pio.renderers.default = "iframe"

# Semantic surface colours (dataviz reference palette, light mode).
_SEM_COLORS = {"GroundSurface": "#898781", "RoofSurface": "#eb6834",
               "WallSurface": "#2a78d6"}


def visualize_record(record, title=None):
    """Plotly figure of one `draw_samples` record: the reconstructed surfaces.

    Faces are shaded and outlined by semantic class; vertices are markers
    carrying their graph node id, so a malformed generated ring is visible.
    Returns a `go.Figure` -- call `.show()` in a notebook or `.write_html(path)`.
    """
    import plotly.graph_objects as go  # heavy import; only needed for plots

    cj = record["cityjson"]
    verts = np.asarray(cj["vertices"], dtype=float)
    obj_id, obj = next(iter(cj["CityObjects"].items()))
    geom = obj["geometry"][0]
    surfaces = geom["semantics"]["surfaces"]

    by_type = {}
    for face, sem in zip(geom["boundaries"][0], geom["semantics"]["values"][0]):
        by_type.setdefault(surfaces[sem]["type"], []).append(face[0])

    fig = go.Figure()
    for stype, rings in by_type.items():
        color = _SEM_COLORS.get(stype, "#eda100")
        i, j, k = [], [], []
        xs, ys, zs = [], [], []
        for ring in rings:
            # ponytail: fan triangulation, only correct for star-shaped rings --
            # switch to earcut on the fitted plane if concave faces show up.
            for t in range(1, len(ring) - 1):
                i.append(ring[0])
                j.append(ring[t])
                k.append(ring[t + 1])
            loop = verts[ring + ring[:1]]
            xs += [*loop[:, 0], None]
            ys += [*loop[:, 1], None]
            zs += [*loop[:, 2], None]
        fig.add_trace(go.Mesh3d(
            x=verts[:, 0], y=verts[:, 1], z=verts[:, 2], i=i, j=j, k=k,
            color=color, opacity=0.5, flatshading=True,
            name=f"{stype} ({len(rings)})", showlegend=True, hoverinfo="name",
        ))
        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs, mode="lines", line=dict(color=color, width=3),
            showlegend=False, hoverinfo="skip",
        ))

    fig.add_trace(go.Scatter3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2], mode="markers",
        marker=dict(size=3, color="#0b0b0b"), name="vertices",
        hovertext=[f"v{n}<br>x={p[0]:.2f} y={p[1]:.2f} z={p[2]:.2f}"
                   for n, p in enumerate(verts)],
        hoverinfo="text", showlegend=False,
    ))

    n_faces = sum(len(r) for r in by_type.values())
    fig.update_layout(
        title=title or f"{obj_id} — {len(verts)} vertices, {n_faces} faces",
        paper_bgcolor="#fcfcfb",
        font=dict(family='system-ui, "Segoe UI", sans-serif', color="#0b0b0b"),
        legend=dict(itemsizing="constant"),
        scene=dict(aspectmode="data"),
        margin=dict(l=0, r=0, t=50, b=0),
    )
    return fig

In [10]:
fig = visualize_record(records[4])
fig.show()

In [11]:
pos, node_labels, edge_labels = model.sample(batch_size=10)

In [12]:
def visualize_sample(pos, node_labels, edge_labels, index=0, model=None):
    """Levi-graph view of one instance of a raw `model.sample(...)` batch.

    OFF nodes are dropped; face nodes are drawn at their members' centroid
    (`levi_figure`'s convention), not at the coords the model generated for them.
    Pass `model` to denormalize back to metres. Returns a `go.Figure`.
    """
    import torch

    from src.dataset.dataset import EDGE_OFF, OFF
    from src.visualize_levi import levi_figure

    p = pos[index]
    if model is not None:
        p = torch.as_tensor(model._denormalize_coords(p))
    p, nl = p.detach().cpu(), node_labels[index].detach().cpu()
    el = edge_labels[index].detach().cpu()

    keep = (nl != OFF).nonzero().flatten()
    p, nl, el = p[keep], nl[keep], el[keep][:, keep]
    ei = (el != EDGE_OFF).nonzero().t()          # [2, E], both directions
    return levi_figure({"id": f"sample_{index}", "x": p, "node_labels": nl,
                        "edge_index": ei, "edge_attr": el[ei[0], ei[1]]})

In [13]:
fig2 = visualize_sample(pos, node_labels, edge_labels, 7, model)
fig2.show()

## Diagnostics

Throwaway exploratory cells — no TDD gate, no reproducibility contract. Added to
explain the levi-1 gap: the wandb dashboard looks fine but the samples do not.

Reads `model`, `datamodule`, `cfg` and `visualize_record` from the cells above.

| | Question | What the answer means |
|---|---|---|
| **D1** | Reconstruct a *real* graph from timestep `t0` instead of from the prior. | Good at small `t0`, degrading smoothly → the denoiser works, the failure is the high-`t` regime. Bad even at `t0=10` → the denoiser itself. |
| **D2** | Structural statistics over ~200 samples vs the test split. | Confirms or kills the coordinate-collapse hypothesis (74% of the coord loss targets the origin). |
| **D3** | Overfit 100 buildings, then sample. | Still bad → sampling/posterior bug. Good → capacity or data, not the sampler. |

Note the frame convention used throughout: the chain runs in **scaled, xy-centered**
units. `model._denormalize_coords` returns metres; a *difference* between two tensors
already in that frame needs only `× model.coord_scale`.

In [ ]:
import numpy as np
import pandas as pd
import torch

from src.dataset.dataset import VERTEX, OFF, EDGE_VV, EDGE_VF, graph_collate_fn


def graph_stats(coords, node_labels, edge_labels):
    """Structural summary of one Levi graph -- generated or real, same function.

    Reads the graph tensors directly instead of going through `graph_to_cityjson`,
    so nothing is filtered out by the converter's drop rules: a collapsed sample
    has to show up here rather than silently vanish from the record list.
    Lengths are metres; `coords` must already be denormalized.
    """
    coords = np.asarray(coords, dtype=float)
    v = np.flatnonzero(node_labels == VERTEX)
    f = np.flatnonzero((node_labels != VERTEX) & (node_labels != OFF))

    vf_deg = np.array([(edge_labels[j, v] == EDGE_VF).sum() for j in f], dtype=float)
    vv_deg = (edge_labels[np.ix_(v, v)] == EDGE_VV).sum(1).astype(float) if len(v) else np.zeros(0)
    bbox = (coords[v].max(0) - coords[v].min(0)) if len(v) else np.zeros(3)

    return {
        "n_vertices": len(v),
        "n_faces": len(f),
        "n_off": int((node_labels == OFF).sum()),
        "footprint_diag_m": float(np.linalg.norm(bbox[:2])),
        "height_m": float(bbox[2]),
        "mean_vv_degree": float(vv_deg.mean()) if len(vv_deg) else 0.0,
        "mean_vf_degree": float(vf_deg.mean()) if len(vf_deg) else 0.0,
        # A face node needs >=3 member vertices before it can become a ring at all.
        "frac_faces_usable": float((vf_deg >= 3).mean()) if len(vf_deg) else 0.0,
        "frac_isolated_vertices": float((vv_deg == 0).mean()) if len(vv_deg) else 1.0,
    }


def real_graph_stats(ds, i):
    """`graph_stats` for one dataset item (already metres, uncentered)."""
    item = ds[int(i)]
    if isinstance(item, tuple):
        item = item[0]
    return graph_stats(item["x"].numpy(),
                       item["node_categories"].argmax(-1).numpy(),
                       item["y"].squeeze(-1).numpy())


def gen_graph_stats(model, pos, node_labels, edge_labels, i):
    """`graph_stats` for slot `i` of a chain output (scaled -> metres)."""
    return graph_stats(model._denormalize_coords(pos[i]),
                       node_labels[i].cpu().numpy(),
                       edge_labels[i].cpu().numpy())


STAT_COLS = ["n_vertices", "n_faces", "footprint_diag_m", "height_m",
             "mean_vv_degree", "mean_vf_degree", "frac_faces_usable",
             "frac_isolated_vertices"]

In [ ]:
@torch.no_grad()
def reconstruct_from(model, batch, t0):
    """D1 -- run the reverse chain seeded at `t0` from a *real* graph, not the prior.

    Noises a clean batch to `t0` with the training forward process, then runs the
    identical reverse loop `sample()` uses, from `t0` down to 0. Slot identity is
    preserved end to end (the chain never permutes nodes), so the output compares
    to the target slot-for-slot.

    Everything comes back in the model's scaled, xy-centered frame.
    """
    model.eval()
    dev = model.device
    batch = {k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in batch.items()}
    R0, X0, E0, net_mask = model._prepare(batch)
    B = R0.shape[0]

    z = model.noise.apply_noise(
        R0, X0, E0, net_mask,
        t_int=torch.full((B, 1), t0, dtype=torch.long, device=dev))
    pos, X_t, E_t = z["pos_t"], z["X_t"], z["E_t"]

    for s_int in reversed(range(0, t0)):
        t_int = torch.full((B, 1), s_int + 1, dtype=torch.long, device=dev)
        s_arr = torch.full((B, 1), s_int, dtype=torch.long, device=dev)
        R_pred, E_pred, X_pred = model(X_t, pos, E_t, t_int, node_mask=net_mask)
        pos, X_t, E_t = model.noise.sample_zs_from_zt_and_pred(
            pos_t=pos, X_t=X_t, E_t=E_t,
            pred_pos=R_pred, pred_X=X_pred, pred_E=E_pred,
            t_int=t_int, s_int=s_arr, node_mask=net_mask)

    return {"pos": pos, "node_labels": X_t.argmax(-1), "edge_labels": E_t.argmax(-1),
            "pos_true": R0, "node_labels_true": X0.argmax(-1),
            "edge_labels_true": E0.argmax(-1)}

In [ ]:
from torch.utils.data import DataLoader

recon_batch = next(iter(DataLoader(datamodule.test_dataset, batch_size=16,
                                   shuffle=False, collate_fn=graph_collate_fn)))


def is_face(labels):
    """Face nodes are every real node that is not a vertex (ground/roof/wall)."""
    return (labels != VERTEX) & (labels != OFF)


def _prf(pred_mask, true_mask):
    """Precision/recall of one boolean class mask; nan when the class is absent.

    Aggregate accuracy is useless here -- OFF is 74% of node slots and EDGE_OFF is
    98.5% of edge slots, so both are ~1.0 regardless of whether the minority
    classes were recovered at all.
    """
    tp = (pred_mask & true_mask).sum().item()
    return (tp / pred_mask.sum().item() if pred_mask.any() else float("nan"),
            tp / true_mask.sum().item() if true_mask.any() else float("nan"))


rows = []
for t0 in (10, 50, 100, 250, model.T):
    out = reconstruct_from(model, recon_batch, t0)
    nl, nl_t = out["node_labels"], out["node_labels_true"]
    el, el_t = out["edge_labels"], out["edge_labels_true"]
    off_diag = ~torch.eye(el.shape[1], dtype=torch.bool, device=el.device)

    row = {"t0": t0}

    # Coordinates, split by node role: face slots are diffused and carry coord
    # loss, but `graph_to_cityjson` ignores them entirely -- so face error is a
    # separate signal from the vertex error that actually reaches the geometry.
    # Both tensors are in the same scaled frame, so the difference needs only coord_scale.
    d_m = (out["pos"] - out["pos_true"]) * model.coord_scale
    row["vertex_rmse_m"] = d_m[nl_t == VERTEX].pow(2).mean().sqrt().item()
    row["face_rmse_m"] = d_m[is_face(nl_t)].pow(2).mean().sqrt().item()

    for name, pred, true in (("vertex", nl == VERTEX, nl_t == VERTEX),
                             ("face", is_face(nl), is_face(nl_t))):
        row[f"{name}_prec"], row[f"{name}_rec"] = _prf(pred, true)

    for name, cls in (("vv", EDGE_VV), ("vf", EDGE_VF)):
        row[f"{name}_prec"], row[f"{name}_rec"] = _prf((el == cls) & off_diag,
                                                       (el_t == cls) & off_diag)

    row["n_vert_pred"] = (nl == VERTEX).sum(1).float().mean().item()
    row["n_vert_true"] = (nl_t == VERTEX).sum(1).float().mean().item()
    rows.append(row)

recon_sweep = pd.DataFrame(rows).set_index("t0")
recon_sweep.round(3)

In [ ]:
@torch.no_grad()
def single_step_error(model, batch, ts):
    """One forward pass per `t` -- no reverse chain, no error accumulation.

    Splits the reconstruction error into what the network gets wrong in a *single*
    x0 prediction versus what the chain piles on top. The reverse posterior is
    verified exact (feeding an oracle x0 prediction reconstructs perfectly at every
    t0, positions and classes alike), so whatever shows up here is the network alone.

    Two baselines make the columns readable:
      copy_input_m  -- RMSE of just returning the noised input. ~0 at small t, so a
                       model scoring worse than this is losing to the identity map.
      predict_com_m -- RMSE of predicting the centre of mass (zeros in this frame),
                       i.e. the best possible t->T guess. A ceiling, not a target:
                       the network should never be worse than this at any t.
    """
    model.eval()
    dev = model.device
    batch = {k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in batch.items()}
    R0, X0, E0, net_mask = model._prepare(batch)
    B, N = R0.shape[0], R0.shape[1]
    nl_t, el_t = X0.argmax(-1), E0.argmax(-1)
    off_diag = ~torch.eye(N, dtype=torch.bool, device=dev)
    v_true, f_true = nl_t == VERTEX, is_face(nl_t)

    def rmse(pred, mask):
        return ((pred - R0)[mask] * model.coord_scale).pow(2).mean().sqrt().item()

    rows = []
    for t in ts:
        z = model.noise.apply_noise(
            R0, X0, E0, net_mask,
            t_int=torch.full((B, 1), t, dtype=torch.long, device=dev))
        R_pred, E_pred, X_pred = model(z["X_t"], z["pos_t"], z["E_t"],
                                       z["t_int"], node_mask=net_mask)
        nl_p, el_p = X_pred.argmax(-1), E_pred.argmax(-1)

        row = {"t": t,
               "vertex_rmse_m": rmse(R_pred, v_true),
               "face_rmse_m": rmse(R_pred, f_true),
               "copy_input_m": rmse(z["pos_t"], v_true),
               "predict_com_m": rmse(torch.zeros_like(R0), v_true)}
        for name, pred, true in (("vertex", nl_p == VERTEX, v_true),
                                 ("face", is_face(nl_p), f_true)):
            row[f"{name}_prec"], row[f"{name}_rec"] = _prf(pred, true)
        for name, cls in (("vv", EDGE_VV), ("vf", EDGE_VF)):
            row[f"{name}_prec"], row[f"{name}_rec"] = _prf((el_p == cls) & off_diag,
                                                           (el_t == cls) & off_diag)
        rows.append(row)
    return pd.DataFrame(rows).set_index("t")


single_step = single_step_error(
    model, recon_batch, [1, 10, 25, 50, 100, 150, 200, 250, 300, 350, 400, 450, 500])
single_step.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8), dpi=140)

ax.plot(single_step.index, single_step["vertex_rmse_m"], color=C_GEN, linewidth=2,
        marker="o", markersize=5, label="network, single step (vertices)")
ax.plot(single_step.index, single_step["copy_input_m"], color=INK_MUTED, linewidth=1.5,
        linestyle="--")
ax.axhline(single_step["predict_com_m"].iloc[0], color=INK_MUTED, linewidth=1.5,
           linestyle=":")

# Reference lines are baselines, not series -- direct-labelled rather than given
# their own hue, so the one real series keeps sole ownership of colour.
ax.annotate("copy the noised input", (single_step.index[-3], single_step["copy_input_m"].iloc[-3]),
            color=INK_MUTED, fontsize=8, ha="right", va="bottom")
ax.annotate("predict the centre of mass", (single_step.index[1], single_step["predict_com_m"].iloc[0]),
            color=INK_MUTED, fontsize=8, ha="left", va="bottom")

ax.set_xlabel("t (diffusion timestep)", color=INK, fontsize=9)
ax.set_ylabel("vertex RMSE [m]", color=INK, fontsize=9)
ax.set_title("Single-step x0 error vs the two trivial predictors",
             color=INK, fontsize=10, loc="left")
ax.legend(frameon=False, fontsize=9, labelcolor=INK, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#c9c9c4")
ax.grid(axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(colors=INK_MUTED, labelsize=8)
fig.patch.set_facecolor(SURFACE)
ax.set_facecolor(SURFACE)
fig.tight_layout()
plt.show()

# The read. copy_input_m is ~0 at t=1 BY CONSTRUCTION (alpha_bar = 1, so the noised
# input *is* the answer) -- no model can beat it there and a ratio test against it is
# meaningless. What matters is the absolute floor, and where the curves cross.
lo = single_step.loc[1]
beats = single_step.index[(single_step["vertex_rmse_m"] < single_step["copy_input_m"]).values]
print(f"floor at t=1:       {lo['vertex_rmse_m']:.3f} m")
print(f"CoM ceiling:        {lo['predict_com_m']:.3f} m  "
      f"(the optimal prediction as t->T, so tracking it at high t is healthy)")
print(f"beats copy-input:   t >= {beats.min() if len(beats) else 'never'}")

In [ ]:
@torch.no_grad()
def error_decomposition(model, batch, t0s, single_step=None):
    """Split chain vertex error into a rigid shift and a shape residual.

    Two things this answers that the raw RMSE cannot:

    1. IS THE SHIFT THE CoM LEAK?  Under se2 the zero-CoM projection covers ALL
       n_max slots in xy, so sum(real) == -sum(off). OFF slots are pinned to 0 in
       the target but are free in the chain, so any drift there *forces* the real
       centroid to -sum(off)/n_real. That prediction is exact, and comparing it to
       the observed xy shift confirms or kills the mechanism outright.
    2. IS THE SHAPE ERROR COMPOUNDING?  se2 quotients absolute xy out on purpose --
       a building shifted 0.9 m with correct shape is a good building. So the shape
       residual, not the total, is what should be compared against the single-step
       error at the same t. That ratio is the exposure-bias factor.

    Aggregation: sums of squares are pooled across graphs and rooted once, and the
    shift is vertex-count weighted. Averaging per-graph RMSEs first would break the
    identity below by Jensen and silently reweight small graphs.

        total^2 == translation^2 / 3 + shape^2      (exact, asserted)
    """
    rows = []
    for t0 in t0s:
        out = reconstruct_from(model, batch, t0)
        v = out["node_labels_true"] == VERTEX
        off = out["node_labels_true"] == OFF
        d = (out["pos"] - out["pos_true"]) * model.coord_scale        # metres

        sq_total = sq_shape = sq_shift = 0.0
        n_ent = n_vert = 0
        obs_xy, pred_xy = [], []
        for b in range(d.shape[0]):
            db = d[b][v[b]]
            n_v = int(v[b].sum())
            if not n_v:
                continue
            shift = db.mean(0)
            sq_total += db.pow(2).sum().item()
            sq_shape += (db - shift).pow(2).sum().item()
            sq_shift += n_v * shift.pow(2).sum().item()
            n_ent += db.numel()
            n_vert += n_v
            obs_xy.append(shift[:2])
            # Exact forced translation: whatever the off slots hold in xy, the real
            # nodes must cancel it to keep the all-slot mean at zero.
            pred_xy.append(-(out["pos"][b][off[b]] * model.coord_scale).sum(0)[:2] / n_v)

        total = (sq_total / n_ent) ** 0.5
        shape = (sq_shape / n_ent) ** 0.5
        trans = (sq_shift / n_vert) ** 0.5
        obs_xy, pred_xy = torch.stack(obs_xy), torch.stack(pred_xy)

        rows.append({
            "t0": t0,
            "total_rmse_m": total,
            "shape_rmse_m": shape,
            "translation_m": trans,
            "xy_shift_observed_m": obs_xy.pow(2).sum(-1).mean().sqrt().item(),
            "xy_shift_predicted_m": pred_xy.pow(2).sum(-1).mean().sqrt().item(),
            # cosine between observed and CoM-predicted shift: ~1 means the leak
            # explains the direction too, not just the magnitude.
            "leak_alignment": torch.cosine_similarity(obs_xy, pred_xy, dim=-1).mean().item(),
            "off_slot_drift_m": (out["pos"][off] * model.coord_scale).norm(dim=-1).mean().item(),
        })

    df = pd.DataFrame(rows).set_index("t0")
    resid = (df["total_rmse_m"] ** 2 - df["translation_m"] ** 2 / 3 - df["shape_rmse_m"] ** 2).abs()
    assert (resid / df["total_rmse_m"] ** 2 < 1e-6).all(), "decomposition identity violated"

    if single_step is not None:
        df["single_step_m"] = single_step["vertex_rmse_m"].reindex(df.index)
        df["compounding"] = df["shape_rmse_m"] / df["single_step_m"]
    return df


decomp = error_decomposition(model, recon_batch, [10, 50, 100, 250], single_step)
print(decomp.round(3).to_string())

r = decomp.loc[10]
print(f"\nCoM leak:     observed xy shift {r['xy_shift_observed_m']:.3f} m  vs  "
      f"predicted from off-slot drift {r['xy_shift_predicted_m']:.3f} m  "
      f"(alignment {r['leak_alignment']:+.2f})")
print("              -> " + ("CONFIRMED: off-slot drift is translating the building; "
                             "an off-slot anchor is the fix"
                             if abs(r["xy_shift_observed_m"] - r["xy_shift_predicted_m"])
                             < 0.3 * r["xy_shift_observed_m"] and r["leak_alignment"] > 0.7
                             else "NOT the leak: the shift comes from somewhere else"))
if "compounding" in decomp:
    print(f"\nShape error is {r['compounding']:.1f}x the single-step error at t=10 "
          f"({r['shape_rmse_m']:.3f} m vs {r['single_step_m']:.3f} m).")
    print("              -> >3x means the chain leaves the noised-input manifold "
          "(exposure bias); ~1x means the loss mask / capacity is the whole story.")

In [ ]:
from src.post_process.post_process import graph_to_cityjson


def record_from(model, pos, node_labels, edge_labels, i, tag="rec"):
    """Pack slot `i` of a chain output into a `visualize_record`-shaped record.

    Returns None when the converter rejects the graph -- which is itself a result.
    """
    coords = model._denormalize_coords(pos[i])
    nl = node_labels[i].cpu().numpy()
    el = edge_labels[i].cpu().numpy()
    cj = graph_to_cityjson(coords, nl, el, building_id=f"{tag}_{i}")
    return None if not cj else {"coords": coords, "node_labels": nl,
                                "edge_labels": el, "cityjson": cj}


T0, IDX = 50, 0
out = reconstruct_from(model, recon_batch, T0)
truth = record_from(model, out["pos_true"], out["node_labels_true"],
                    out["edge_labels_true"], IDX, tag="true")
rec = record_from(model, out["pos"], out["node_labels"], out["edge_labels"], IDX)

print(f"converted -> ground truth: {truth is not None}, reconstruction: {rec is not None}")
if truth:
    visualize_record(truth, title=f"ground truth (slot {IDX})").show()
if rec:
    visualize_record(rec, title=f"reconstructed from t0={T0}").show()

In [ ]:
N_BATCHES, BATCH = 10, 20          # ~200 buildings; the reverse chain is the cost here

gen_rows = []
for b in range(N_BATCHES):
    p, nl, el = model.sample(batch_size=BATCH)
    gen_rows += [gen_graph_stats(model, p, nl, el, i) for i in range(p.shape[0])]
    print(f"sampled batch {b + 1}/{N_BATCHES}", end="\r")
gen_df = pd.DataFrame(gen_rows)

# Reference: raw test graphs, no converter in the way, so the comparison is
# apples-to-apples with gen_df (which is also pre-converter).
ref_idx = np.random.default_rng(0).choice(len(datamodule.test_dataset), 400, replace=False)
real_df = pd.DataFrame([real_graph_stats(datamodule.test_dataset, i) for i in ref_idx])

summary = pd.concat({"generated": gen_df[STAT_COLS].median(),
                     "real": real_df[STAT_COLS].median()}, axis=1)
summary["gen/real"] = summary["generated"] / summary["real"].replace(0, np.nan)
summary

In [ ]:
import matplotlib.pyplot as plt

# The notebook's two categorical hues, validated as a pair: normal-vision dE 33.6,
# worst-CVD dE 24.7, both well clear of the floor. Identity is also carried by the
# legend and the dotted median rules, never by colour alone.
C_GEN, C_REAL = "#eb6834", "#2a78d6"
INK, INK_MUTED, GRID, SURFACE = "#0b0b0b", "#3d3d3a", "#ececeb", "#fcfcfb"


def compare_hist(col, unit="m", bins=40):
    """Generated vs real distribution of one statistic, overlaid.

    Shared bin edges clipped at the 99th percentile -- a few runaway generated
    values would otherwise squash the real distribution into one bar.
    """
    fig, ax = plt.subplots(figsize=(6.5, 3.4), dpi=140)
    lo = min(gen_df[col].min(), real_df[col].min())
    hi = max(gen_df[col].quantile(0.99), real_df[col].quantile(0.99))
    edges = np.linspace(lo, hi if hi > lo else lo + 1, bins)

    for k, (df, color, label) in enumerate(((real_df, C_REAL, "real (test)"),
                                            (gen_df, C_GEN, "generated"))):
        ax.hist(df[col], bins=edges, density=True, color=color, alpha=0.4)
        ax.hist(df[col], bins=edges, density=True, color=color,
                histtype="step", linewidth=2, label=label)
        med = df[col].median()
        ax.axvline(med, color=color, linewidth=2, linestyle=":")
        ax.annotate(f"median {med:.2g}", (med, ax.get_ylim()[1] * (0.93 - 0.13 * k)),
                    color=INK_MUTED, fontsize=8, ha="center",
                    bbox=dict(fc=SURFACE, ec="none", pad=1.5))

    ax.set_xlabel(f"{col} [{unit}]" if unit else col, color=INK, fontsize=9)
    ax.set_ylabel("density", color=INK_MUTED, fontsize=9)
    ax.legend(frameon=False, fontsize=9, labelcolor=INK)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#c9c9c4")
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK_MUTED, labelsize=8)
    fig.patch.set_facecolor(SURFACE)
    ax.set_facecolor(SURFACE)
    fig.tight_layout()
    return fig


# The headline check: if generated footprints sit at a fraction of the real median,
# the off-slot dilution in coord_loss is confirmed.
for col, unit in (("footprint_diag_m", "m"), ("height_m", "m"),
                  ("n_vertices", ""), ("mean_vf_degree", "")):
    compare_hist(col, unit)
plt.show()

In [ ]:
# Does the sampler land on the class marginals it was trained toward, or does it
# over-produce the majority classes (OFF / EDGE_OFF)?
p, nl, el = model.sample(batch_size=128)
off_diag = ~torch.eye(el.shape[1], dtype=torch.bool, device=el.device)

node_cmp = pd.DataFrame({
    "generated": [(nl == c).float().mean().item() for c in range(model.num_node_classes)],
    "train_marginal": model.noise.X_marginals.cpu().numpy(),
}, index=["vertex", "ground", "roof", "wall", "off"])

edge_cmp = pd.DataFrame({
    "generated": [(el[:, off_diag] == c).float().mean().item()
                  for c in range(model.num_edge_classes)],
    "train_marginal": model.noise.E_marginals.cpu().numpy(),
}, index=["off", "vv", "vf"])

for name, df in (("node classes", node_cmp), ("edge classes", edge_cmp)):
    df["ratio"] = df["generated"] / df["train_marginal"]
    print(f"\n{name}\n{df.round(4)}")

In [ ]:
import copy

import lightning as L
from torch.utils.data import DataLoader, Subset

# Warm-start from the trained weights rather than a fresh init: the question is
# "can this architecture + sampler emit a good building at all", and warm-starting
# answers it in minutes instead of hours. For the stricter version, swap in
# CityJSONDiffusionModule(**dict(model.hparams)) and raise max_epochs.
overfit = copy.deepcopy(model)
overfit.train()
for mod in overfit.modules():
    if isinstance(mod, torch.nn.Dropout):
        mod.p = 0.0
overfit.lr, overfit.lr_scheduler = 3e-4, "none"

SMALL_IDX = list(range(100))
small_loader = DataLoader(Subset(datamodule.train_dataset, SMALL_IDX), batch_size=20,
                          shuffle=True, collate_fn=graph_collate_fn)

L.Trainer(max_epochs=400, accelerator="gpu", devices=1, logger=False,
          enable_checkpointing=False, enable_model_summary=False,
          log_every_n_steps=5).fit(overfit, small_loader)

In [ ]:
overfit = overfit.to(model.device).eval()
p, nl, el = overfit.sample(batch_size=20)

ov_df = pd.DataFrame([gen_graph_stats(overfit, p, nl, el, i) for i in range(p.shape[0])])
target_df = pd.DataFrame([real_graph_stats(datamodule.train_dataset, i) for i in SMALL_IDX])

print(pd.concat({"overfit samples": ov_df[STAT_COLS].median(),
                 "the 100 targets": target_df[STAT_COLS].median()}, axis=1))

shown = 0
for i in range(p.shape[0]):
    r = record_from(overfit, p, nl, el, i, tag="overfit")
    if r:
        visualize_record(r, title=f"overfit sample {i}").show()
        shown += 1
        if shown == 2:
            break

print(f"\n{shown}/{p.shape[0]} overfit samples converted.")
if shown == 0:
    print("None converted despite memorized data -> the sampler/posterior is the "
          "problem, not capacity or the dataset.")